In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-6-luna"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [2]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, reasoning_effort="none", messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
#
#
#############  TOOLS EXAMPE
#
# 1. define a function
#

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
def get_ticket_price(dst:str)->str:
    print(f"Tool called for city {dst}")
    price = ticket_prices.get(dst.lower(), "Unknown ticket price")
    return f"The price of a ticket to {dst} is {price}"
#
#
# 2. create a dictionary that describe the tools (fixed format)
#
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "dst": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["dst"],
        "additionalProperties": False
    }
}

#
#
# 3. define the list of tools to send to the LLM
#
tools = [{"type": "function", "function": price_function}]


In [ ]:
#
# 4. send the message with the history and list of tools and detect if we get back
#    a request to use a tool or not
#
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, reasoning_effort="none", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, reasoning_effort="none", messages=messages)
    
    return response.choices[0].message.content

In [6]:
#
# 3/4. The function to call the tool
#      of course this should be available in the function above.. but we put it here
# 
#   Here is not a super smart way to call a function.. matching the name using if..
#   you just probably reference the function and send the parameters in one go (no if..)  
def handle_tool_calls(message):
    responses = []
    #
    # note that we loop through the tool calls that the LLm wants to call
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses